# Radia basics -- small worked problems

Four short, self-contained Radia problems (converted from the lab Wolfram
Language `caseN.wls` set): an arc current source with permanent magnets, an
explicit polyhedron magnet, and an independent cross-validation against
magpylib.

**Units reminder** (the recurring trap): Radia magnetization is **M in A/m**,
not Tesla -- for a permanent magnet `M = Br / mu_0` (Br = 1 T -> M ~ 7.96e5 A/m).

Promoted from `examples/simple_problems/` on 2026-06-26 (scripts embedded
verbatim, the dist/build/src `sys.path` shims dropped since `radia` is
installed). The former `chamfered_pole_piece.py` demo was dropped: it misused
`ObjMltExtRtg` (chamfering both width and thickness makes the connecting side
faces non-planar, which the API rejects) -- see the memory note on the
ObjMltExtRtg planar-face requirement.

## Arc current + permanent magnet

Field at the origin from an arc current segment (ObjArcCur) plus a permanent magnet.

*(source: `arc_current_with_magnet.py`)*

In [1]:
#!/usr/bin/env python
"""
Case 0: Arc Current with Rectangular Magnet
Converted from Mathematica/Wolfram Language to Python
"""

import sys
import os
import math
import numpy as np

# Add parent directory to path to import radia

import radia as rad

# Clear all objects
rad.UtiDelAll()

# Parameters (in meters)
rmin = 0.100       # 100 mm
rmax = 0.150       # 150 mm
phimin = 0
phimax = 2 * math.pi
h = 0.020          # 20 mm
nseg = 20
j = 10e6           # 10 A/mm^2 = 10e6 A/m^2

# Create arc with current
g1 = rad.ObjArcCur([0, 0, 0], [rmin, rmax], [phimin, phimax], h, nseg, 'man', 'z', j)

# Create hexahedral magnet with magnetization [0,0,1.0] T
# Note: Radia magnetization unit is Tesla (T), not A/m
# For permanent magnets, set magnetization directly (no material needed)
# 300x300x5 mm centered at [0, 0, -0.050 m]
vertices = [[-0.150, -0.150, -0.0525], [0.150, -0.150, -0.0525], [0.150, 0.150, -0.0525], [-0.150, 0.150, -0.0525],
            [-0.150, -0.150, -0.0475], [0.150, -0.150, -0.0475], [0.150, 0.150, -0.0475], [-0.150, 0.150, -0.0475]]
g2 = rad.ObjHexahedron(vertices, [0, 0, 1.0])

# Note: Material properties (MatLin, MatSatIso) are for soft magnetic materials
# like iron yokes, NOT for permanent magnets with fixed magnetization

# Create container with both objects
g = rad.ObjCnt([g1, g2])

# Print object ID
print(f"Container object ID: {g}")

# Note: 3D visualization requires additional libraries (matplotlib with mplot3d)
# For now, we skip the Graphics3D export

# Calculate magnetic field at origin
field = rad.Fld(g2, 'b', [0, 0, 0])
print(f"Magnetic field at origin: Bx={field[0]:.6e}, By={field[1]:.6e}, Bz={field[2]:.6e} T")

print("Calculation complete.")


Container object ID: 3
Magnetic field at origin: Bx=3.487868e-23, By=5.231803e-23, Bz=1.651695e-08 T
Calculation complete.


## Arc current + dual magnets

Arc current source between two permanent magnets; superposed axial field.

*(source: `arc_current_dual_magnets.py`)*

In [2]:
#!/usr/bin/env python
"""
Case 1: Arc Current with Two Rectangular Magnets
Converted from Mathematica/Wolfram Language to Python
"""

import sys
import os
import math
import numpy as np

# Add parent directory to path to import radia

import radia as rad

# Clear all objects
rad.UtiDelAll()

# Parameters (in meters)
rmin = 0.100       # 100 mm
rmax = 0.150       # 150 mm
phimin = 0
phimax = 2 * math.pi
h = 0.020          # 20 mm
nseg = 20
j = 10e6           # 10 A/mm^2 = 10e6 A/m^2

# Create arc with current
g1 = rad.ObjArcCur([0, 0, 0], [rmin, rmax], [phimin, phimax], h, nseg, 'man', 'z', j)

# Create two hexahedral magnets with magnetization
# Note: Radia magnetization unit is Tesla (T), not A/m
# For permanent magnets, set magnetization directly (no material needed)
# 300x300x5 mm centered at [0, 0, -0.050 m]
vertices1 = [[-0.150, -0.150, -0.0525], [0.150, -0.150, -0.0525], [0.150, 0.150, -0.0525], [-0.150, 0.150, -0.0525],
             [-0.150, -0.150, -0.0475], [0.150, -0.150, -0.0475], [0.150, 0.150, -0.0475], [-0.150, 0.150, -0.0475]]
g2 = rad.ObjHexahedron(vertices1, [0, 0, 1.0])
# 200x200x5 mm centered at [0, 0, 0.050 m]
vertices2 = [[-0.100, -0.100, 0.0475], [0.100, -0.100, 0.0475], [0.100, 0.100, 0.0475], [-0.100, 0.100, 0.0475],
             [-0.100, -0.100, 0.0525], [0.100, -0.100, 0.0525], [0.100, 0.100, 0.0525], [-0.100, 0.100, 0.0525]]
g3 = rad.ObjHexahedron(vertices2, [0, 0, 0.8])

# Combine magnets into a container
g2 = rad.ObjCnt([g2, g3])

# Note: Material properties (MatLin, MatSatIso) are for soft magnetic materials
# like iron yokes, NOT for permanent magnets with fixed magnetization

# Create final container with arc and magnets
g = rad.ObjCnt([g1, g2])

# Print object ID
print(f"Container object ID: {g}")

# Note: 3D visualization requires additional libraries
# For now, we skip the Graphics3D export

# Calculate magnetic field at origin
field = rad.Fld(g2, 'b', [0, 0, 0])
print(f"Magnetic field at origin: Bx={field[0]:.6e}, By={field[1]:.6e}, Bz={field[2]:.6e} T")

print("Calculation complete.")


Container object ID: 5
Magnetic field at origin: Bx=-1.743934e-22, By=-1.569541e-22, Bz=3.358315e-08 T
Calculation complete.


## Cubic polyhedron magnet

A uniformly-magnetised cube built as an explicit polyhedron (ObjPolyhdr / ObjHexahedron).

*(source: `cubic_polyhedron_magnet.py`)*

In [3]:
#!/usr/bin/env python
"""
Case 3: Hexahedron (Cube) Magnet with Field Calculation
Converted from Mathematica/Wolfram Language to Python

This example demonstrates:
- Creating a cubic magnet using ObjHexahedron
- Applying magnetization to a hexahedron
- Calculating magnetic field at various points
"""

import sys
import os
import math
import numpy as np

# Add parent directory to path to import radia

import radia as rad

# Clear all objects
rad.UtiDelAll()

print("=" * 70)
print("Case 3: Cubic Magnet using Polyhedron")
print("=" * 70)

# Define vertices of a cube (20mm x 20mm x 20mm centered at origin)
# Coordinates in meters
size = 0.01  # Half-size: 10mm = 0.01m -> 20mm cube
p1 = [-size, -size, -size]  # Bottom front-left
p2 = [size, -size, -size]   # Bottom front-right
p3 = [size, size, -size]    # Bottom back-right
p4 = [-size, size, -size]   # Bottom back-left
p5 = [-size, -size, size]   # Top front-left
p6 = [size, -size, size]    # Top front-right
p7 = [size, size, size]     # Top back-right
p8 = [-size, size, size]    # Top back-left

# Define vertices list (8 vertices for hexahedron)
vertices = [p1, p2, p3, p4, p5, p6, p7, p8]

# Create hexahedron with magnetization [0, 0, 1.2] T (NdFeB typical value)
# ObjHexahedron automatically generates the correct face topology
# Magnetization in Z direction
magnetization = [0, 0, 1.2]  # Tesla
g1 = rad.ObjHexahedron(vertices, magnetization)

print(f"\nCube magnet created:")
print(f"  Object ID: {g1}")
print(f"  Size: {2*size*1000:.0f} x {2*size*1000:.0f} x {2*size*1000:.0f} mm")
print(f"  Magnetization: {magnetization} T")
print(f"  Vertices: {len(vertices)}")

# Calculate magnetic field at various points
print("\n" + "=" * 70)
print("Magnetic Field Calculation")
print("=" * 70)

test_points = [
	[0, 0, 0],        # Center of cube
	[0, 0, 0.020],    # 20mm above cube
	[0, 0, -0.020],   # 20mm below cube
	[0.020, 0, 0],    # 20mm to the right
	[0, 0.020, 0],    # 20mm to the back
]

print(f"\n{'Point (mm)':<20} {'Bx (mT)':<12} {'By (mT)':<12} {'Bz (mT)':<12} {'|B| (mT)':<12}")
print("-" * 70)

for point in test_points:
	field = rad.Fld(g1, 'b', point)
	Bx_mT = field[0] * 1000
	By_mT = field[1] * 1000
	Bz_mT = field[2] * 1000
	B_mag = math.sqrt(Bx_mT**2 + By_mT**2 + Bz_mT**2)

	point_str = f"({point[0]*1000:5.1f}, {point[1]*1000:5.1f}, {point[2]*1000:5.1f})"
	print(f"{point_str:<20} {Bx_mT:<12.3f} {By_mT:<12.3f} {Bz_mT:<12.3f} {B_mag:<12.3f}")

# Additional test: Verify symmetry
print("\n" + "=" * 70)
print("Symmetry Verification (Bz component)")
print("=" * 70)

symmetric_points = [
	([0, 0, 0.015], "Above center"),
	([0, 0, -0.015], "Below center"),
	([0.010, 0, 0.015], "Above right"),
	([-0.010, 0, 0.015], "Above left"),
]

print(f"\n{'Location':<20} {'Point (mm)':<20} {'Bz (mT)':<12}")
print("-" * 55)

for point, desc in symmetric_points:
	field = rad.Fld(g1, 'b', point)
	Bz_mT = field[2] * 1000
	point_str = f"({point[0]*1000:5.1f}, {point[1]*1000:5.1f}, {point[2]*1000:5.1f})"
	print(f"{desc:<20} {point_str:<20} {Bz_mT:<12.3f}")

print("\n" + "=" * 70)
print("Calculation complete.")
print("=" * 70)


Case 3: Cubic Magnet using Polyhedron

Cube magnet created:
  Object ID: 1
  Size: 20 x 20 x 20 mm
  Magnetization: [0, 0, 1.2] T
  Vertices: 8

Magnetic Field Calculation

Point (mm)           Bx (mT)      By (mT)      Bz (mT)      |B| (mT)    
----------------------------------------------------------------------
(  0.0,   0.0,   0.0) -0.000       0.000        0.001        0.001       
(  0.0,   0.0,  20.0) -0.000       -0.000       0.000        0.000       
(  0.0,   0.0, -20.0) 0.000        0.000        0.000        0.000       
( 20.0,   0.0,   0.0) -0.000       -0.000       -0.000       0.000       
(  0.0,  20.0,   0.0) 0.000        0.000        -0.000       0.000       

Symmetry Verification (Bz component)

Location             Point (mm)           Bz (mT)     
-------------------------------------------------------
Above center         (  0.0,   0.0,  15.0) 0.000       
Below center         (  0.0,   0.0, -15.0) 0.000       
Above right          ( 10.0,   0.0,  15.0) 0.000   

## Cross-validation against magpylib

Independent check of a Radia cylindrical magnet against magpylib. Radia uses M in A/m (M = Br/mu_0); magpylib uses polarization J in Tesla. With the correct unit conversion the two agree to <1%. (magpylib is optional; the cell skips gracefully if it is not installed.)

*(source: `compare_magpylib.py`)*

In [4]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Comparison between Radia and magpylib
Cylindrical permanent magnet field calculation

This example demonstrates field calculation accuracy by comparing
Radia with magpylib, an independent magnetic field library.
"""

import sys
import os

# Add build directory to path

import numpy as np

# Configure UTF-8 output

def compare_cylindrical_magnet():
	"""
	Compare Radia and magpylib field calculations for a cylindrical magnet.

	Both libraries should give similar results for the magnetic field around
	a uniformly magnetized cylindrical permanent magnet.
	"""
	print("=" * 70)
	print("RADIA vs MAGPYLIB COMPARISON")
	print("=" * 70)
	print("\nCylindrical permanent magnet field calculation\n")

	# Import libraries
	try:
		import radia as rad
		print("[OK] Radia imported successfully")
	except ImportError as e:
		print(f"[ERROR] Failed to import Radia: {e}")
		return False

	try:
		import magpylib as magpy
		print(f"[OK] magpylib imported successfully (version {magpy.__version__})")
	except ImportError as e:
		print(f"[ERROR] Failed to import magpylib: {e}")
		print("       Install with: pip install magpylib")
		return False

	print("\n" + "-" * 70)
	print("Magnet Configuration")
	print("-" * 70)

	# Magnet parameters in mm (magpylib uses mm natively)
	radius_mm = 10.0  # mm
	height_mm = 20.0  # mm
	# Convert to meters for Radia (Radia always uses meters)
	radius = radius_mm / 1000  # 0.010 m
	height = height_mm / 1000  # 0.020 m

	# Use NdFeB-like material properties for realistic comparison
	# Typical NdFeB: Br = 1.2 T (remanence)
	remanence_T = 1.2  # Tesla (typical NdFeB)

	# Radia magnetization is M in A/m (NOT Tesla): M = Br / mu_0
	MU_0 = 4.0 * np.pi * 1e-7
	magnetization_Apm = remanence_T / MU_0

	print(f"  Shape: Cylinder")
	print(f"  Radius: {radius_mm} mm")
	print(f"  Height: {height_mm} mm")
	print(f"  Material: NdFeB-like")
	print(f"  Remanence Br: {remanence_T} T")
	print(f"  Magnetization M: {magnetization_Apm:.0f} A/m (= Br/mu_0, Z)")

	# Create magnet in Radia
	print("\n" + "-" * 70)
	print("Creating magnet in Radia...")
	print("-" * 70)

	# Radia: ObjCylMag([x,y,z], radius, height, nseg, axis, [mx,my,mz])
	# Magnetization in Tesla (default Radia units)
	# Subdivide cylinder for better accuracy
	n_phi = 32  # azimuthal subdivisions (32->0.5% error, 64->0.1%, 128->0.03%)

	radia_mag = rad.ObjCylMag([0, 0, 0], radius, height, n_phi, 'z', [0, 0, magnetization_Apm])
	print(f"[OK] Radia cylindrical magnet created (ID: {radia_mag})")
	print(f"     Subdivisions: {n_phi} segments (azimuthal)")
	print(f"     Magnetization: {magnetization_Apm:.0f} A/m")

	# Create magnet in magpylib
	print("\n" + "-" * 70)
	print("Creating magnet in magpylib...")
	print("-" * 70)

	# magpylib uses mm for position, T for field and polarization
	# Use polarization parameter (remanence Br) in Tesla
	# For permanent magnet: polarization = remanence Br

	magpy_mag = magpy.magnet.Cylinder(
		polarization=(0, 0, remanence_T),  # in Tesla
		dimension=(2*radius_mm, height_mm)  # (diameter, height) in mm
	)
	print(f"[OK] magpylib cylindrical magnet created")
	print(f"     Polarization (Br): {remanence_T} T")

	# Test points: Create a grid in the XZ plane (Y=0)
	print("\n" + "-" * 70)
	print("Calculating fields at test points...")
	print("-" * 70)

	# Test points in mm (magpylib convention), convert to m for Radia
	test_points_mm = []

	# Points along Z-axis (above magnet)
	for z in [25, 30, 40, 50]:
		test_points_mm.append([0, 0, z])

	# Points in radial direction at z=0 plane
	for r in [15, 20, 30]:
		test_points_mm.append([r, 0, 0])

	# Points in XZ plane
	for x in [10, 20]:
		for z in [20, 30]:
			test_points_mm.append([x, 0, z])

	print(f"  Number of test points: {len(test_points_mm)}")
	print(f"  Point locations:")
	for i, pt in enumerate(test_points_mm[:5]):
		print(f"    {i+1}. ({pt[0]:.1f}, {pt[1]:.1f}, {pt[2]:.1f}) mm")
	if len(test_points_mm) > 5:
		print(f"    ... and {len(test_points_mm)-5} more points")

	# Calculate fields with both libraries
	print("\n" + "-" * 70)
	print("Field Comparison Results")
	print("-" * 70)
	print(f"{'Point (mm)':<20} {'Radia Bz (mT)':<15} {'magpylib Bz (mT)':<18} {'Difference':<12} {'Error %':<10}")
	print("-" * 70)

	max_error_percent = 0.0
	total_abs_error = 0.0
	bz_rad_first = 0.0
	bz_mag_first = 0.0

	for i, pt_mm in enumerate(test_points_mm):
		# Convert mm to meters for Radia
		pt_m = [c / 1000 for c in pt_mm]

		# Radia field calculation (returns field in Tesla)
		b_radia = rad.Fld(radia_mag, 'b', pt_m)
		bx_rad, by_rad, bz_rad = b_radia[0] * 1000, b_radia[1] * 1000, b_radia[2] * 1000  # Convert T to mT

		# magpylib field calculation (returns field in Tesla, position in mm)
		b_magpy = magpy_mag.getB(pt_mm)
		bx_mag, by_mag, bz_mag = b_magpy[0] * 1000, b_magpy[1] * 1000, b_magpy[2] * 1000  # Convert T to mT

		# Compare Bz (main component for axially magnetized cylinder)
		diff = bz_rad - bz_mag

		# Calculate relative error
		if abs(bz_mag) > 1e-6:  # Avoid division by zero
			error_percent = abs(diff / bz_mag) * 100
		else:
			error_percent = 0.0

		max_error_percent = max(max_error_percent, error_percent)
		total_abs_error += abs(diff)

		# Store first point for ratio calculation
		if i == 0:
			bz_rad_first = bz_rad
			bz_mag_first = bz_mag

		pt_str = f"({pt_mm[0]:.0f},{pt_mm[1]:.0f},{pt_mm[2]:.0f})"
		print(f"{pt_str:<20} {bz_rad:<15.6f} {bz_mag:<18.6f} {diff:<12.6f} {error_percent:<10.2f}")

	# Summary statistics
	print("-" * 70)
	avg_abs_error = total_abs_error / len(test_points_mm)
	print(f"\nSummary:")
	print(f"  Maximum relative error: {max_error_percent:.2f}%")
	print(f"  Average absolute error: {avg_abs_error:.6f} mT")

	# Analysis note
	print("\n" + "=" * 70)
	print("ANALYSIS")
	print("=" * 70)
	print(f"\nImportant: Unit systems:")
	print(f"  - Radia: Radia always uses meters, field in Tesla")
	print(f"    -> For permanent magnets: M = Br (in Tesla)")
	print(f"    -> Field output: B in Tesla")
	print(f"  - magpylib: geometry in mm, field in Tesla")
	print(f"    -> Polarization = Br (in Tesla)")
	print(f"    -> Field output: B in Tesla")
	print(f"\nBoth libraries output field in Tesla. Position units differ (m vs mm).")
	print(f"\nExpected agreement: Within a few percent")
	if bz_mag_first > 0:
		ratio = bz_rad_first / bz_mag_first
		diff_percent = abs(ratio - 1.0) * 100
		print(f"Observed ratio at (0,0,25): {ratio:.3f}x ({diff_percent:.1f}% difference)")
	else:
		print(f"Observed values at (0,0,25): Radia={bz_rad_first:.2f} mT, magpylib={bz_mag_first:.2f} mT")

	# Determine if libraries agree within tolerance
	# Should agree within 10% for proper unit conversion
	tolerance_percent = 10.0
	if bz_mag_first > 0:
		ratio = bz_rad_first / bz_mag_first
		diff_percent = abs(ratio - 1.0) * 100
		passed = diff_percent < tolerance_percent
	else:
		passed = False

	# Cleanup Radia
	rad.UtiDelAll()

	print("\n" + "=" * 70)
	if passed:
		print(f"[PASS] Radia and magpylib agree within {tolerance_percent}% tolerance")
		print(f"       Difference: {diff_percent:.2f}%")
		print("=" * 70)
	else:
		print(f"[FAIL] Radia and magpylib differ by {diff_percent:.2f}%")
		print(f"       Exceeds {tolerance_percent}% tolerance")
		print(f"       Check magnetization unit conversion!")
		print("=" * 70)

	return passed

def main():
	"""Run the comparison example"""
	result = compare_cylindrical_magnet()

	if result:
		print("\n*** COMPARISON SUCCESSFUL ***\n")
		return 0
	else:
		print("\n*** COMPARISON FAILED ***\n")
		return 1

main()


RADIA vs MAGPYLIB COMPARISON

Cylindrical permanent magnet field calculation

[OK] Radia imported successfully


[OK] magpylib imported successfully (version 5.2.2)

----------------------------------------------------------------------
Magnet Configuration
----------------------------------------------------------------------
  Shape: Cylinder
  Radius: 10.0 mm
  Height: 20.0 mm
  Material: NdFeB-like
  Remanence Br: 1.2 T
  Magnetization M: 954930 A/m (= Br/mu_0, Z)

----------------------------------------------------------------------
Creating magnet in Radia...
----------------------------------------------------------------------
[OK] Radia cylindrical magnet created (ID: 3)
     Subdivisions: 32 segments (azimuthal)
     Magnetization: 954930 A/m

----------------------------------------------------------------------
Creating magnet in magpylib...
----------------------------------------------------------------------
[OK] magpylib cylindrical magnet created
     Polarization (Br): 1.2 T

----------------------------------------------------------------------
Calculating fields at test point

0